In [1]:
# Install all required libraries
!pip install rank_bm25 sentence-transformers chromadb groq wikipedia-api langchain langchain-groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!pip install wikipedia -q
import wikipedia
print("wikipedia imported successfully")

  Preparing metadata (setup.py) ... done
wikipedia imported successfully


In [5]:
import os
from google.colab import userdata

# Load Groq API key from Colab secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Core imports
import wikipedia
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from groq import Groq
import numpy as np

print("All imports successful.")

All imports successful.


In [6]:
# Topics to fetch
topics = [
    "Retrieval-augmented generation",
    "Large language model",
    "Transformer (machine learning model)",
    "BERT (language model)",
    "Attention mechanism (artificial intelligence)"
]

raw_docs = []

for topic in topics:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        raw_docs.append({
            "title": page.title,
            "content": page.content
        })
        print(f"Fetched: {page.title}")
    except Exception as e:
        print(f"Failed: {topic} — {e}")

print(f"\nTotal articles fetched: {len(raw_docs)}")

Fetched: Retrieval-augmented generation
Fetched: Large language model
Fetched: Transformer (deep learning)
Fetched: BERT (language model)
Failed: Attention mechanism (artificial intelligence) — Page id "Attention mechanism (artificial intelligence)" does not match any pages. Try another id!

Total articles fetched: 4


In [7]:
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Splits a long text into smaller overlapping chunks by word count.

    chunk_size=500: each chunk has 500 words
    overlap=50: next chunk starts 50 words before previous chunk ended
    — this prevents context from being cut off at boundaries
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # slide forward by 450 words, not 500

    return chunks


all_chunks = []
chunk_metadata = []

for doc in raw_docs:
    chunks = chunk_text(doc["content"], chunk_size=500, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "source": doc["title"],  # which article this chunk came from
            "chunk_index": i         # position of this chunk within that article
        })

print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 51


In [8]:
# BM25 requires tokenized input (list of word lists, not raw strings)
tokenized_chunks = [chunk.lower().split() for chunk in all_chunks]

bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks.")

BM25 index built over 51 chunks.


In [9]:
# Embedding function using all-MiniLM-L6-v2
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# In-memory ChromaDB client — no disk storage needed for this notebook
client = chromadb.Client()
collection = client.create_collection(
    name="week5_hybrid",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # cosine similarity for sentence embeddings
)

# Add chunks in batches to avoid memory spike
batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch_chunks = all_chunks[i:i+batch_size]
    batch_ids = [str(j) for j in range(i, i+len(batch_chunks))]
    batch_meta = chunk_metadata[i:i+len(batch_chunks)]

    collection.add(
        documents=batch_chunks,
        ids=batch_ids,
        metadatas=batch_meta
    )

print(f"ChromaDB collection built: {collection.count()} chunks indexed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB collection built: 51 chunks indexed.


In [10]:
def bm25_search(query, top_k=10):
    """
    Returns top_k chunks ranked by BM25 keyword score.

    Steps:
    1. Tokenize the query the same way we tokenized chunks (lowercase + split)
    2. BM25 scores every chunk in the corpus against the query
    3. Sort by score descending, return top_k
    """
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    # argsort gives indices from lowest to highest — [::-1] reverses to highest first
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "text": all_chunks[idx],
            "score": scores[idx],
            "index": int(idx),
            "source": chunk_metadata[idx]["source"]
        })

    return results


# Quick test
test = bm25_search("what is attention mechanism", top_k=3)
for r in test:
    print(f"Score: {r['score']:.3f} | Source: {r['source']}")
    print(r['text'][:150])
    print("---")

Score: 3.829 | Source: BERT (language model)
The small model aims to fool the large model. DeBERTa (2020) is a significant architectural variant, with disentangled attention. Its key idea is to t
---
Score: 3.635 | Source: BERT (language model)
smaller datasets to optimize its performance on specific tasks such as natural language inference and text classification, and sequence-to-sequence-ba
---
Score: 3.605 | Source: Transformer (deep learning)
and each layer in a transformer model has multiple attention heads. While each attention head attends to the tokens that are relevant to each token, m
---


In [11]:
def semantic_search(query, top_k=10):
    """
    Returns top_k chunks ranked by cosine similarity via ChromaDB.

    Unlike BM25, this understands meaning — so "how models understand context"
    can match chunks about "attention" even without exact word overlap.
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    output = []
    for i in range(len(results["documents"][0])):
        output.append({
            "text": results["documents"][0][i],
            # ChromaDB returns distance (lower = more similar)
            # convert to similarity: 1 - distance
            "score": 1 - results["distances"][0][i],
            "index": int(results["ids"][0][i]),  # cast string ID back to int
            "source": results["metadatas"][0][i]["source"]
        })

    return output


# Quick test
test = semantic_search("how does self-attention work", top_k=3)
for r in test:
    print(f"Score: {r['score']:.3f} | Source: {r['source']}")
    print(r['text'][:150])
    print("---")

Score: 0.467 | Source: Transformer (deep learning)
and each layer in a transformer model has multiple attention heads. While each attention head attends to the tokens that are relevant to each token, m
---
Score: 0.443 | Source: Transformer (deep learning)
an order of magnitude fewer parameters than LSTMs. One of its authors, Jakob Uszkoreit, suspected that attention without recurrence would be sufficien
---
Score: 0.426 | Source: Transformer (deep learning)
W V {\displaystyle W^{K},W^{V}} , thus: MultiQueryAttention ( Q , K , V ) = Concat i ∈ [ n heads ] ( Attention ( X W i Q , X W K , X W V ) ) W O {\dis
---


In [12]:
def reciprocal_rank_fusion(bm25_results, semantic_results, k=60):
    """
    Merges two ranked lists into one using RRF.

    Formula: score = 1 / (k + rank) for each list, summed per document.

    Why k=60: it dampens the impact of top ranks — rank 1 vs rank 2
    difference is small, preventing one list from dominating the other.
    A chunk appearing in both lists gets two contributions — so it naturally
    rises to the top.
    """
    rrf_scores = {}   # chunk index → combined RRF score
    chunk_lookup = {} # chunk index → chunk data (text, source etc.)

    # Process BM25 results
    for rank, result in enumerate(bm25_results):
        idx = result["index"]
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)
        chunk_lookup[idx] = result

    # Process semantic results
    for rank, result in enumerate(semantic_results):
        idx = result["index"]
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)
        chunk_lookup[idx] = result

    # Sort by combined RRF score descending
    sorted_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

    fused_results = []
    for idx in sorted_indices:
        result = chunk_lookup[idx].copy()
        result["rrf_score"] = rrf_scores[idx]
        fused_results.append(result)

    return fused_results


def hybrid_search(query, top_k=10):
    bm25_results = bm25_search(query, top_k=top_k)
    semantic_results = semantic_search(query, top_k=top_k)
    fused = reciprocal_rank_fusion(bm25_results, semantic_results)
    return fused[:top_k]


# Quick test
test = hybrid_search("BERT pretraining objective", top_k=3)
for r in test:
    print(f"RRF Score: {r['rrf_score']:.4f} | Source: {r['source']}")
    print(r['text'][:150])
    print("---")

RRF Score: 0.0164 | Source: BERT (language model)
if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for ta
---
RRF Score: 0.0164 | Source: BERT (language model)
smaller datasets to optimize its performance on specific tasks such as natural language inference and text classification, and sequence-to-sequence-ba
---
RRF Score: 0.0161 | Source: Transformer (deep learning)
difficulty in converging. In the original paper, the authors recommended using learning rate warmup. That is, the learning rate should linearly scale 
---


In [13]:
# Cross-encoder: sees query + document together — more accurate than bi-encoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


In [14]:
def rerank(query, candidates, top_k=5):
    """
    Takes hybrid search results and reranks them using cross-encoder.

    Cross-encoder sees (query + document) together as one input — this is
    why it's more accurate than bi-encoder. It can model direct interaction
    between query words and document words, not just vector similarity.

    retrieve_k should be larger than final top_k — cast a wide net first,
    then let the reranker pick the best ones precisely.
    """
    if not candidates:
        return []

    # Create (query, document) pairs — cross-encoder scores each pair
    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.predict(pairs)

    # Attach reranker score to each candidate
    for i, candidate in enumerate(candidates):
        candidate["rerank_score"] = float(scores[i])

    # Sort by reranker score descending
    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_k]


def hybrid_search_with_rerank(query, retrieve_k=20, final_k=5):
    """
    Full pipeline:
    Step 1 — Hybrid search retrieves top retrieve_k (wide net, fast)
    Step 2 — Reranker picks best final_k from those (precise, slow)

    retrieve_k=20, final_k=5 is a standard production ratio.
    """
    candidates = hybrid_search(query, top_k=retrieve_k)
    reranked = rerank(query, candidates, top_k=final_k)
    return reranked


# Quick test
test = hybrid_search_with_rerank("how does BERT use masked language modeling", retrieve_k=20, final_k=3)
for r in test:
    print(f"Rerank Score: {r['rerank_score']:.4f} | Source: {r['source']}")
    print(r['text'][:200])
    print("---")

Rerank Score: 5.9422 | Source: BERT (language model)
layer, translating a one-hot vector into a dense vector based on its token type. Position: The position embeddings are based on a token's position in the sequence. BERT uses absolute position embeddin
---
Rerank Score: 5.5520 | Source: BERT (language model)
if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for tasks like question answering or document classifica
---
Rerank Score: 5.5412 | Source: BERT (language model)
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-
---


In [15]:
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def generate_answer(query, context_chunks, model="llama-3.1-8b-instant"):
    """
    Takes query + top reranked chunks → generates a grounded answer via Groq.

    temperature=0.1 — keeps the answer factual, not creative.
    The prompt explicitly tells the model to only use the provided context
    — this is what prevents hallucination in RAG systems.
    """
    context = "\n\n".join([
        f"[Source: {c['source']}]\n{c['text']}"
        for c in context_chunks
    ])

    prompt = f"""You are a precise question-answering assistant.
Answer the question using ONLY the context provided below.
If the context does not contain the answer, say "I don't know based on the provided context."

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=512
    )

    return response.choices[0].message.content


# End-to-end test
query = "What is the role of attention mechanism in transformers?"
context = hybrid_search_with_rerank(query, retrieve_k=20, final_k=5)
answer = generate_answer(query, context)

print(f"QUERY: {query}\n")
print(f"ANSWER:\n{answer}")

QUERY: What is the role of attention mechanism in transformers?

ANSWER:
The attention mechanism in transformers enables the model to process relationships between all elements in a sequence simultaneously, regardless of their distance from each other. It calculates "soft" weights for each token, more precisely for its embedding, by using multiple attention heads, each with its own "relevance" for calculating its own soft weights. This allows the model to capture more complex and long-range dependencies in deeper layers and to attend to the tokens that are relevant to each token for different definitions of "relevance".


In [17]:
def compare_retrieval_methods(query, top_k=3):
    """
    Runs all three methods on the same query and prints results side by side.

    Purpose: visually see where each method wins and loses.
    - BM25 wins on exact keyword matches
    - Semantic wins on conceptual/paraphrased queries
    - Hybrid + Rerank wins on most queries by combining both
    """
    print(f"QUERY: {query}")
    print("=" * 70)

    # BM25
    print("\n[BM25 — Keyword Only]")
    bm25_results = bm25_search(query, top_k=top_k)
    for i, r in enumerate(bm25_results):
        print(f"  Rank {i+1} | Score: {r['score']:.3f} | Source: {r['source']}")
        print(f"  {r['text'][:150]}...")

    # Semantic
    print("\n[Semantic — Embedding Only]")
    sem_results = semantic_search(query, top_k=top_k)
    for i, r in enumerate(sem_results):
        print(f"  Rank {i+1} | Score: {r['score']:.3f} | Source: {r['source']}")
        print(f"  {r['text'][:150]}...")

    # Hybrid + Rerank
    print("\n[Hybrid + Rerank — Full Pipeline]")
    hybrid_results = hybrid_search_with_rerank(query, retrieve_k=20, final_k=top_k)
    for i, r in enumerate(hybrid_results):
        print(f"  Rank {i+1} | Rerank Score: {r['rerank_score']:.4f} | Source: {r['source']}")
        print(f"  {r['text'][:150]}...")

    print("\n" + "=" * 70)


# Three queries — each designed to expose a different method's weakness
compare_retrieval_methods("BERT masked language model pretraining")  # exact term → BM25 should win
compare_retrieval_methods("how do neural networks understand context") # conceptual → semantic should win
compare_retrieval_methods("transformer self-attention mechanism explanation") # both → hybrid should win

QUERY: BERT masked language model pretraining

[BM25 — Keyword Only]
  Rank 1 | Score: 9.604 | Source: Transformer (deep learning)
  difficulty in converging. In the original paper, the authors recommended using learning rate warmup. That is, the learning rate should linearly scale ...
  Rank 2 | Score: 6.693 | Source: Large language model
  compound model is then fine-tuned on an image-text dataset. This basic construction can be applied with more sophistication to improve the model. The ...
  Rank 3 | Score: 6.651 | Source: BERT (language model)
  if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for ta...

[Semantic — Embedding Only]
  Rank 1 | Score: 0.716 | Source: BERT (language model)
  if the second sentence is a valid continuation of the first one. This helps BERT understand relationships between sentences, which is important for ta...
  Rank 2 | Score: 0.680 | Source: BERT (language 

DAY 1A — HYBRID SEARCH + RERANKING
Advanced RAG | Month 1, Week 5

WHAT IS THIS NOTEBOOK?
This notebook implements an advanced RAG retrieval pipeline that goes beyond basic semantic search. It combines three retrieval strategies and benchmarks them against each other on the same queries.

WHAT WE BUILT:
1. BM25 Search — keyword-based retrieval using term frequency and inverse document frequency. No embeddings, purely statistical.

2. Semantic Search — embedding-based retrieval using all-MiniLM-L6-v2 (384-dim) stored in ChromaDB with HNSW index and cosine similarity.

3. Hybrid Search — combined BM25 + Semantic using Reciprocal Rank Fusion (RRF, k=60) to merge ranked lists without worrying about score scale differences.

4. Reranking — cross-encoder (ms-marco-MiniLM-L-6-v2) that sees query + document together for precise final ranking. Two-stage pipeline: retrieve top-20, rerank to top-5.

5. Answer Generation — Groq (llama-3.1-8b-instant) generates grounded answers from the top reranked chunks only.

CORPUS:
4 Wikipedia articles — RAG, LLMs, Transformers, BERT
Chunked into 51 chunks (500 words, 50-word overlap)

WHAT WE ACHIEVED:
- Hybrid + Rerank outperformed both BM25 and semantic alone on exact and mixed queries
- Identified that vague queries fail at retrieval level — no ranking method fixes a bad query (motivation for Day 1B)
- Negative reranker scores correctly flagged low-confidence retrievals — reranker acts as a quality gate, not just a sorter
- Observed that noisy corpus (Wikipedia math markup) hurts all methods equally — preprocessing is a real concern

KEY NUMBERS:
BM25 top score:      9.604  (exact keyword match)
Semantic top score:  0.716  (cosine similarity)
Reranker top score:  5.94   (cross-encoder confidence)
Reranker low score: -2.17   (actively flagged as irrelevant)

